## SCD Type 2 with Azure Databricks

### Task 1.1 - Creating the Unity Catalog Schema for this Lab

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS banking_lab

### Task 1.2 - Creating the `dim_customer` SCD Type 2 Table

This table tracks **historical changes** in customer data using the SCD Type 2 pattern.

Key concepts:
- Each change creates a **new version of the record**
- Old versions are **retained for history and auditing**

Important columns:
- `customer_sk` → surrogate key (auto-generated)
- `customer_id` → business key
- `valid_from`, `valid_to`, `is_current` → track record history

We also:
- Apply **Liquid Clustering on `customer_id`** for faster lookups
- Enable **Change Data Feed (CDF)** for auditing changes

This table is critical for **regulatory reporting and point-in-time analysis**.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS banking_lab.dim_customer (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id STRING NOT NULL,
    full_name STRING,
    email STRING,
    city STRING,
    segment STRING,
    account_type STRING,
    valid_from TIMESTAMP NOT NULL,
    valid_to TIMESTAMP NOT NULL,
    is_current BOOLEAN
)
USING DELTA
CLUSTER BY (customer_id)
TBLPROPERTIES (
  delta.enableChangeDataFeed = true
);

### Task 1.3 - Create the `fact_transactions` table

This table stores all financial transactions (credits, debits, transfers).

Key design decisions:
- One row per transaction
- Linked to customer via `customer_id`

We apply:
- **Liquid Clustering on (`customer_id`, `transaction_date`)**
  → optimizes queries filtering by customer and time
- **Change Data Feed (CDF)** for audit tracking

This table will support:
- Transaction analytics
- Fraud detection
- Regulatory audit requirements (FCA, Basel III)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS banking_lab.fact_transactions (
    transaction_id STRING NOT NULL,
    account_id STRING,
    customer_id STRING,
    transaction_type STRING,
    amount DECIMAL(15,2),
    currency STRING DEFAULT 'GBP',
    transaction_date DATE,
    description STRING
)
USING DELTA
CLUSTER BY (customer_id, transaction_date)
TBLPROPERTIES (
  delta.enableChangeDataFeed = true
);

### Task 2.1 - Load the Initial Customer Records

We insert the first batch of customer records into the dimension table.

Each record is:
- Marked as `is_current = true`
- Given an open-ended validity (`valid_to = 9999-12-31`)

This represents the **initial snapshot of customers** before any changes occur.

From here, we will simulate real-world updates.



In [ ]:
from pyspark.sql.functions import lit, to_timestamp

initial_customers = [
    ("C-1001", "Emma Hartley",      "emma.hartley@northbank.com",     "London",     "Retail",   "Savings"),
    ("C-1002", "James Weston",      "james.weston@northbank.com",     "Manchester", "Retail",   "Current"),
    ("C-1003", "Sophia Chen",       "sophia.chen@northbank.com",      "Edinburgh",  "Premium",  "Savings"),
    ("C-1004", "Oliver Banks",      "oliver.banks@northbank.com",     "Birmingham", "Retail",   "Current"),
    ("C-1005", "Aisha Patel",       "aisha.patel@northbank.com",      "London",     "Business", "Business"),
    ("C-1006", "Liam Murray",       "liam.murray@northbank.com",      "Leeds",      "Retail",   "Savings"),
    ("C-1007", "Charlotte Wright",  "charlotte.wright@northbank.com", "Bristol",    "Premium",  "Current"),
    ("C-1008", "Noah Thompson",     "noah.thompson@northbank.com",    "Glasgow",    "Retail",   "Savings"),
]

df_initial = spark.createDataFrame(
    initial_customers,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)

df_initial = (
    df_initial
    .withColumn("valid_from",  to_timestamp(lit("2020-01-01 00:00:00")))
    .withColumn("valid_to",    to_timestamp(lit("9999-12-31 00:00:00")))
    .withColumn("is_current",  lit(True))
)

df_initial.write.mode("append").saveAsTable("banking_lab.dim_customer")

In [ ]:
%sql
SELECT customer_sk, customer_id, full_name, city, segment, is_current
FROM banking_lab.dim_customer
ORDER BY customer_sk;

### Task 2.2 - Load Staging Data with Customer Changes

The operations team has received the following change notifications:

| Customer | Change |
|----------|--------|
| C-1001 Emma Hartley | Moved from London → Oxford, upgraded to Premium segment |
| C-1003 Sophia Chen | Relocated from Edinburgh → London |
| C-1006 Liam Murray | Updated email address only (city and segment unchanged) |
| C-1009 Priya Singh | New customer joining from Cardiff, Retail segment |

Run the cell below to create a staging DataFrame and register it as a temp view. You will use it in the next task.

In [ ]:
staging_data = [
    ("C-1001", "Emma Hartley",    "emma.hartley@northbank.com",     "Oxford",   "Premium",  "Savings"),
    ("C-1003", "Sophia Chen",     "sophia.chen@northbank.com",      "London",   "Premium",  "Savings"),
    ("C-1006", "Liam Murray",     "liam.new@northbank.com",         "Leeds",    "Retail",   "Savings"),
    ("C-1009", "Priya Singh",     "priya.singh@northbank.com",      "Cardiff",  "Retail",   "Current"),
]

df_staging = spark.createDataFrame(
    staging_data,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)
df_staging.createOrReplaceTempView("staging_customers")
print("Staging data loaded. Temp view 'staging_customers' is available.")
display(df_staging)

### Task 2.3 — Apply SCD Type 2: Step 1 — Close changed records

When customer attributes change (city, segment, email, etc.),  
we must **expire the current version** of the record.

We do this using a MERGE:
- Match on `customer_id`
- Only consider current records (`is_current = true`)
- If any tracked column changes:
  → set `valid_to = current_timestamp()`
  → set `is_current = false`

This ensures we **preserve history instead of overwriting data**.

In [ ]:
%sql
MERGE INTO banking_lab.dim_customer AS target
USING staging_customers AS source
ON target.customer_id = source.customer_id
AND target.is_current = true

WHEN MATCHED AND (
    target.city <> source.city OR
    target.segment <> source.segment OR
    target.email <> source.email OR
    target.account_type <> source.account_type
)
THEN UPDATE SET
    target.valid_to = current_timestamp(),
    target.is_current = false;

### Task 2.3 — Apply SCD Type 2: Step 2 — Insert new versions and new customers

After closing old records, we insert:
- New versions of updated customers
- Brand-new customers not seen before

Each new row:
- Gets `valid_from = current_timestamp()`
- Gets `valid_to = 9999-12-31`
- Is marked as `is_current = true`

This completes the **SCD Type 2 pattern**:
- Old version → closed
- New version → active

In [ ]:
%sql
INSERT INTO banking_lab.dim_customer (
    customer_id, full_name, email, city, segment, account_type,
    valid_from, valid_to, is_current
)
SELECT
    s.customer_id,
    s.full_name,
    s.email,
    s.city,
    s.segment,
    s.account_type,
    current_timestamp(),
    TIMESTAMP '9999-12-31 00:00:00',
    true
FROM staging_customers s
WHERE NOT EXISTS (
    SELECT 1
    FROM banking_lab.silver.dim_customer d
    WHERE d.customer_id = s.customer_id
    AND d.is_current = true
);

### Task 2.4 - View History

We query all versions of a specific customer.

This allows us to see:
- How attributes changed over time
- When each version was active

This is essential for:
- Auditing
- Historical reporting
- Regulatory compliance

In [ ]:
%sql
SELECT *
FROM banking_lab.dim_customer
WHERE customer_id = 'C-1001'
ORDER BY valid_from ASC;

### Task 2.5 - Point-in-Time Query

We reconstruct the customer state at a specific date.

Logic:
A record is valid at time D if:
- `valid_from <= D`
- `valid_to > D`

This allows us to answer:
👉 “What did the customer profile look like on a past date?”

This is critical for:
- Financial audits
- Compliance reporting
- Historical analytics

In [ ]:
%sql
SELECT *
FROM banking_lab.dim_customer
WHERE valid_from <= TIMESTAMP '2020-06-15 00:00:00'
  AND valid_to > TIMESTAMP '2020-06-15 00:00:00';